In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import time
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical
from skopt.utils import use_named_args
import os
import math
import copy
warnings.filterwarnings("ignore")

# 1. 加载数据
print("正在加载数据...")
df_original = pd.read_excel('../../data/quyi_data_x_names.xlsx')
print(f"原始数据集大小: {df_original.shape}")

# 2. 定义目标变量
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']

# 3. 分离特征和目标
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 4. 分割原始数据为训练集和测试集
# Paper protocol: preserve furnace order; first 900 records train, next 500 test.
if len(X_original) != 1400 or len(y_original) != 1400:
    raise ValueError('Expected 1,400 ordered furnace records (900 train + 500 test).')
X_train, X_test = X_original.iloc[:900].copy(), X_original.iloc[900:].copy()
y_train, y_test = y_original.iloc[:900].copy(), y_original.iloc[900:].copy()

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 5. 数据标准化
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train)
X_test_scaled = scaler_X.transform(X_test)

# 6. 准备PyTorch数据集和数据加载器
class TabularDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        if y is not None:
            self.y = torch.tensor(y, dtype=torch.float32)
        else:
            self.y = None
            
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        else:
            return self.X[idx]

# 7. 定义Tab-Transformer模型
class TabTransformer(nn.Module):
    def __init__(self, input_dim, output_dim, embedding_dim=32, depth=4, 
                 heads=8, attn_dropout=0.1, ff_dropout=0.1):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        
        # 嵌入层
        self.embedding = nn.Linear(input_dim, embedding_dim)
        
        # 位置编码
        self.pos_encoder = nn.Parameter(torch.zeros(1, input_dim, embedding_dim))
        
        # Transformer编码器层
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=heads,
            dim_feedforward=embedding_dim * 4,
            dropout=attn_dropout,
            activation='gelu',
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=depth)
        
        # MLP头部
        self.mlp_head = nn.Sequential(
            nn.Linear(embedding_dim * input_dim, embedding_dim * 2),
            nn.LayerNorm(embedding_dim * 2),
            nn.GELU(),
            nn.Dropout(ff_dropout),
            nn.Linear(embedding_dim * 2, embedding_dim),
            nn.LayerNorm(embedding_dim),
            nn.GELU(),
            nn.Dropout(ff_dropout),
            nn.Linear(embedding_dim, output_dim)
        )
        
    def forward(self, x):
        batch_size = x.size(0)
        
        # 重塑输入
        x = x.unsqueeze(-1)
        
        # 为每个特征创建嵌入
        x = x.repeat(1, 1, self.embedding.in_features)
        x = x * torch.eye(self.embedding.in_features, device=x.device)
        
        # 应用线性嵌入
        x = self.embedding(x)  # shape: [batch_size, num_features, embedding_dim]
        
        # 添加位置编码
        x = x + self.pos_encoder
        
        # 通过Transformer编码器
        x = self.transformer_encoder(x)
        
        # 重塑输出
        x = x.reshape(batch_size, -1)
        
        # 通过MLP头部
        x = self.mlp_head(x)
        
        return x

# 检查可用设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# 创建训练集和验证集用于模型训练
# Ordered inner validation split; search scalers see only the inner training block.
_search_cut = int(len(X_train) * 0.8)
_search_scaler_X = StandardScaler().fit(X_train.iloc[:_search_cut])
_search_scaler_y = StandardScaler().fit(y_train.iloc[:_search_cut])
X_train_split = _search_scaler_X.transform(X_train.iloc[:_search_cut])
X_val = _search_scaler_X.transform(X_train.iloc[_search_cut:])
y_train_split = _search_scaler_y.transform(y_train.iloc[:_search_cut])
y_val = _search_scaler_y.transform(y_train.iloc[_search_cut:])
y_search_val = y_train.iloc[_search_cut:]

dimensions = [
    # 嵌入维度的因子，嵌入维度 = mult_factor * base_dim
    Integer(1, 8, name='mult_factor'),
    Integer(8, 16, name='base_dim'),
    
    # Transformer深度
    Integer(1, 6, name='depth'),
    
    # 两种dropout
    Real(0.0, 0.5, name='attn_dropout'),
    Real(0.0, 0.5, name='ff_dropout'),
    
    # 学习率
    Real(0.0001, 0.01, "log-uniform", name='learning_rate'),
    
    # 批量大小
    Integer(16, 128, name='batch_size'),
    
    # 训练轮数
    Integer(10, 100, name='epochs')
]

# 定义保存最佳模型状态的变量

# 定义目标函数 - 使用测试集每个目标变量的最小R²作为优化目标
# 在目标函数中添加确保批量大小为整数的步骤
@use_named_args(dimensions)
def objective(mult_factor, base_dim, depth, attn_dropout, ff_dropout, learning_rate, batch_size, epochs):
    torch.manual_seed(42)
    np.random.seed(42)
    
    # 确保批量大小是整数
    batch_size = int(batch_size)
    
    # 计算嵌入维度
    embedding_dim = mult_factor * base_dim
    
    # 设定heads数量，确保embedding_dim可以被heads整除
    heads = 1  # 默认设为1，避免因为heads数量导致的维度不匹配问题
    
    # 记录开始时间
    start_time = time.time()
    
    # 组合参数
    print(f"参数: emb_dim={embedding_dim}(={mult_factor}×{base_dim}), depth={depth}, attn_drop={attn_dropout:.2f}, ff_drop={ff_dropout:.2f}, lr={learning_rate:.6f}, batch={batch_size}, epochs={epochs}")
    
    # 创建数据集
    train_dataset = TabularDataset(X_train_split, y_train_split)
    val_dataset = TabularDataset(X_val, y_val)
    
    # 创建数据加载器
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    
    # 创建模型
    input_dim = X_train_scaled.shape[1]
    output_dim = len(target_vars)
    
    model = TabTransformer(
        input_dim=input_dim,
        output_dim=output_dim,
        embedding_dim=embedding_dim,
        depth=depth,
        heads=heads,
        attn_dropout=attn_dropout,
        ff_dropout=ff_dropout
    ).to(device)
    
    # 优化器和损失函数
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
    )
    loss_fn = nn.MSELoss()
    
    # ... 其余代码保持不变 ...
    
    # 训练函数
    def train_epoch(model, loader, optimizer, loss_fn, device):
        model.train()
        epoch_loss = 0
        
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            
            # 前向传播
            optimizer.zero_grad()
            y_pred = model(x)
            loss = loss_fn(y_pred, y)
            
            # 反向传播
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        return epoch_loss / len(loader)
    
    # 验证函数
    def validate(model, loader, loss_fn, device):
        model.eval()
        val_loss = 0
        
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                y_pred = model(x)
                loss = loss_fn(y_pred, y)
                val_loss += loss.item()
        
        return val_loss / len(loader)
    
    # 训练模型
    best_val_loss = float('inf')
    patience = 10
    counter = 0
    current_best_model_state = None
    
    train_losses = []
    val_losses = []
    
    # 减少早停的严格程度，以便更好地训练
    for epoch in range(epochs):
        # 训练
        train_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
        train_losses.append(train_loss)
        
        # 验证
        val_loss = validate(model, val_loader, loss_fn, device)
        val_losses.append(val_loss)
        
        # 更新学习率
        scheduler.step(val_loss)
        
        # 检查是否保存模型
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            counter = 0
            current_best_model_state = copy.deepcopy(model.state_dict())
        else:
            counter += 1
            if counter >= patience:
                break
    
    # 如果有最佳模型状态，加载它
    if current_best_model_state:
        model.load_state_dict(current_best_model_state)
    
    # 在测试集上进行预测
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch_x, _ in val_loader:
            all_preds.append(model(batch_x.to(device)).cpu().numpy())
    y_pred = _search_scaler_y.inverse_transform(np.vstack(all_preds))
    r2_scores = [r2_score(y_search_val.iloc[:, idx], y_pred[:, idx])
                 for idx in range(len(target_vars))]
    min_r2 = min(r2_scores)
    avg_r2 = np.mean(r2_scores)
    print(f"  Internal validation minimum R2: {min_r2:.4f}, mean R2: {avg_r2:.4f}")
    end_time = time.time()
    print(f"Time taken: {end_time - start_time:.4f}")
    return -min_r2


print("Bayesian optimization uses training-only validation; the holdout is reserved for final evaluation.")
result = gp_minimize(
    objective,
    dimensions=dimensions,
    n_calls=30,  # 迭代次数
    n_random_starts=10,  # 随机搜索起点数
    random_state=42,
    verbose=True,
    n_jobs=1  # TabTransformer是神经网络，不支持并行
)

# 修改提取最佳参数的代码
best_embedding_dim = int(result.x[0] * result.x[1])
best_mult_factor = int(result.x[0])
best_base_dim = int(result.x[1])
best_depth = int(result.x[2])
best_attn_dropout = result.x[3]
best_ff_dropout = result.x[4]
best_learning_rate = result.x[5]
best_batch_size = int(result.x[6])  # 确保批量大小为整数
best_epochs = int(result.x[7])      # 确保轮数为整数


print("使用找到的最佳模型状态")
print("\n最优参数:")
print(f"embedding_dim: {best_embedding_dim} (= {best_mult_factor} × {best_base_dim})")
print(f"heads: 1")
print(f"depth: {best_depth}")
print(f"attn_dropout: {best_attn_dropout}")
print(f"ff_dropout: {best_ff_dropout}")
print(f"lr: {best_learning_rate}")
print(f"batch_size: {best_batch_size}")
print(f"epochs: {best_epochs}")

# 用最优参数创建最终模型
print("\n用最优参数训练最终模型...")
input_dim = X_train_scaled.shape[1]
output_dim = len(target_vars)

# 创建完整训练数据集和数据加载器
full_train_dataset = TabularDataset(X_train_scaled, y_train_scaled)
full_train_loader = DataLoader(full_train_dataset, batch_size=best_batch_size, shuffle=True)
test_dataset = TabularDataset(X_test_scaled)
test_loader = DataLoader(test_dataset, batch_size=best_batch_size)

# 创建最终模型
final_model = TabTransformer(
    input_dim=input_dim,
    output_dim=output_dim,
    embedding_dim=best_embedding_dim,
    depth=best_depth,
    heads=1,  # 固定为1，保持简单
    attn_dropout=best_attn_dropout,
    ff_dropout=best_ff_dropout
).to(device)

# 加载最佳模型权重
optimizer = torch.optim.Adam(final_model.parameters(), lr=best_learning_rate)
loss_fn = nn.MSELoss()
for epoch in range(best_epochs):
    final_model.train()
    for x, y in full_train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_fn(final_model(x), y)
        loss.backward()
        optimizer.step()

test_dataset = TabularDataset(X_test_scaled)
test_loader = DataLoader(test_dataset, batch_size=best_batch_size)
final_model.eval()
all_preds = []

with torch.no_grad():
    for batch in test_loader:
        if isinstance(batch, list):
            batch = batch[0]
        batch = batch.to(device)
        preds = final_model(batch).cpu().numpy()
        all_preds.append(preds)

# 合并批次预测
y_pred_scaled = np.vstack(all_preds)

# 反标准化预测结果
y_pred = scaler_y.inverse_transform(y_pred_scaled)

# 评估最终模型性能
print("\n优化后模型评估指标:")
overall_r2 = 0
overall_rmse = 0
overall_mae = 0
min_r2_value = float('inf')
min_r2_var = ""

for idx, name in enumerate(target_vars):
    # 计算指标
    mse = mean_squared_error(y_test.iloc[:, idx], y_pred[:, idx])
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test.iloc[:, idx], y_pred[:, idx])
    r2 = r2_score(y_test.iloc[:, idx], y_pred[:, idx])
    
    # 累加指标值
    overall_r2 += r2
    overall_rmse += rmse
    overall_mae += mae
    
    # 记录最小R²
    if r2 < min_r2_value:
        min_r2_value = r2
        min_r2_var = name
    
    # 打印每个目标变量的指标
    print(f"\n{name} 指标:")
    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")

# 计算平均指标
overall_r2 /= len(target_vars)
overall_rmse /= len(target_vars)
overall_mae /= len(target_vars)

print("\n总体平均指标:")
print(f"平均 RMSE: {overall_rmse:.4f}")
print(f"平均 MAE: {overall_mae:.4f}")
print(f"平均 R²: {overall_r2:.4f}")
print(f"最小 R²: {min_r2_value:.4f} (变量: {min_r2_var})")

# 输出贝叶斯优化的最佳结果
print("\n贝叶斯优化最佳结果:")
print(f"最佳最小R²: {-result.fun:.4f}")
print(f"优化迭代次数: {len(result.func_vals)}")